# RTE optimisation paper — migrated to AeroMAPS ≥ 1.1

Port of `run_opt_main_B099.ipynb` (branch `optim_backwards`) onto the generic energy
carriers + markets models on `main`.

**What changed, and what did not.** The published problem drives two design variables,
the biofuel and electrofuel blending shares. Underneath, the old model carried five
biofuel pathways at *constant* sub-shares, so the five collapse into one generic
pathway with no loss of fidelity:

| pathway | share | MFSP (€/L) |
|---|---|---|
| hefa_fog | 0.6 % | 0.815488 |
| hefa_others | 12.5 % | 1.052703 |
| ft_msw | 6.6 % | 1.142423 |
| ft_others | 68.9 % | 1.378082 |
| atj | 11.4 % | 1.38668 |

Blend-weighted mean = **1.3195 €/L**, against the 1.31 €/L reported in §3.1.

The electrofuel chain was rederived the same way: electrolysis 0.59 × H₂→fuel 0.74
gives 2.2904 MJ_elec/MJ_fuel, so the grid factor of 205 → 12 gCO₂/kWh maps to
**130.4 → 7.6 gCO₂/MJ**, matching the "130 to 7" of §3.1.

**Validation target.** Section 1 evaluates a single MDA at the ReFuelEU-linear design
point, for which the paper publishes values (Table 3, and the −10.4 Bn€ label on
Figure 9). Reference numbers from `optim_backwards` are inlined for comparison.

## 0. Setup

Run this notebook from its own directory: the config resolves its relative paths
against itself.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gemseo as gm
from gemseo.algos.design_space import DesignSpace
from gemseo.algos.opt.scipy_local.settings.slsqp import SLSQP_Settings

from aeromaps import create_process
from aeromaps.core.gemseo import CustomDataConverter
from aeromaps.utils.functions import custom_logger_config

warnings.filterwarnings("ignore")

# Same call as the published notebooks. GEMSEO's default level is INFO (20), which is
# what prints the optimisation iteration log - lower it to WARNING and the run goes
# silent. custom_logger_config only patches NumPy-docstring parsing and annotates the
# "Args section is missing" warning; it suppresses nothing.
custom_logger_config(gm.configure_logger())

# Share of world traffic departing the EU in 2019 (AeroSCOPE), used to downscale
# the world carbon budget and the world energy allocations.
EU_ASK_SHARE = 15.49 / 100

# ReFuelEU reference years carrying a design variable
OPTIM_YEARS = [2030, 2035, 2040, 2045, 2050]

## 1. Validation — one MDA at the ReFuelEU-linear design point

`config_rte.yaml` keeps the paper's four markets. `config_1m.yaml` collapses the three
passenger markets into one; because the scenario gives every passenger market the same
CAGR and the same efficiency gain, the two agree to 7 significant figures while the
collapsed one carries 90 disciplines instead of 100.

In [ ]:
def build_process(config="config_rte.yaml", optimisation=False, carbon_budget_share=2.6):
    """Create the process and set every scenario parameter.

    The constraint models of `constraints_rte.py` are part of the chain even for a
    plain MDA, so their inputs must always be defined - otherwise GEMSEO cannot
    order the disciplines.
    """
    process = create_process(configuration_file=config, optimisation=optimisation)

    # Entry point for the airfare <-> RPK loop, as in the published notebook.
    process.parameters.price_elasticity = -0.9
    process.parameters.airfare_per_rpk = pd.Series(
        0.09251431471704129,
        index=range(process.parameters.historic_start_year, process.parameters.end_year + 1),
    )

    # Constraint enforcement years (G2-G6)
    for name in [
        "blend_completeness_constraint",
        "biofuel_use_growth_constraint",
        "electrofuel_use_growth_constraint",
    ]:
        setattr(process.parameters, f"{name}_enforcement_years", OPTIM_YEARS)
    process.parameters.generic_biomass_availability_constraint_enforcement_years = OPTIM_YEARS
    process.parameters.generic_electricity_constraint_enforcement_years = OPTIM_YEARS

    # Ramp-up limits: 0.2 EJ/yr and 20 %/yr, downscaled to the EU perimeter (Eq. 12).
    process.parameters.volume_ramp_up_constraint_biofuel = 0.2 * EU_ASK_SHARE
    process.parameters.rate_ramp_up_constraint_biofuel = 0.2
    process.parameters.volume_ramp_up_constraint_electrofuel = 0.2 * EU_ASK_SHARE
    process.parameters.rate_ramp_up_constraint_electrofuel = 0.2

    # Carbon budget (G1), as a share of the world aviation budget.
    process.parameters.aviation_carbon_budget_objective = carbon_budget_share * EU_ASK_SHARE

    # Fixed leading mandate entries (2020, 2025); later years are the design variables.
    process.parameters.generic_biofuel_mandate_share_values_fixed = [0.0, 2.0]
    process.parameters.generic_electrofuel_mandate_share_values_fixed = [0.0, 0.0]
    return process


def set_mandate(process, biofuel, electrofuel):
    """Impose a mandate for the five reference years, bypassing the optimiser."""
    process.parameters.generic_biofuel_mandate_share_values_optim = biofuel
    process.parameters.generic_electrofuel_mandate_share_values_optim = electrofuel


process = build_process()
print(f"{len(process.mda_chain.disciplines)} disciplines")

In [ ]:
# ReFuelEU mandate, linear interpolation - the 'ReFuelEU (linear)' column of Table 3.
# Values are for 2030..2050; 2020 and 2025 come from the *_fixed parameters.
set_mandate(
    process, biofuel=[4.8, 15.0, 24.0, 27.0, 35.0], electrofuel=[1.2, 5.0, 10.0, 15.0, 35.0]
)

start = time.perf_counter()
process.compute()
print(f"MDA in {time.perf_counter() - start:.2f} s")

In [ ]:
vector = process.data["vector_outputs"]

# Cumulative quantities are series here, not floats: read them at the end year.
cumulative_co2 = vector["cumulative_co2_emissions"].loc[2050]
surplus_loss = vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9

# Reference values produced by the same design point on branch optim_backwards.
reference = {
    "Airfare 2050 (EUR/RPK)": (vector["airfare_per_rpk"].loc[2050], 0.100804, "0.101 (Table 3)"),
    "RPK 2050": (vector["rpk"].loc[2050], 2.215949e12, "~2.2e12 (Figure 8)"),
    "Cumulative CO2 (Gt)": (cumulative_co2, 3.865052, "3.87 (Table 3)"),
    "Discounted surplus loss (Bn EUR)": (surplus_loss, -10.3886, "-10.4 (Figure 9)"),
}

rows = []
for name, (migrated, old, published) in reference.items():
    delta = (migrated - old) / abs(old) * 100
    rows.append([name, f"{migrated:.6g}", f"{old:.6g}", f"{delta:+.2f} %", published])

print(
    pd.DataFrame(
        rows, columns=["indicator", "migrated", "optim_backwards", "delta", "paper"]
    ).to_string(index=False)
)

Every indicator should land within a few tenths of a percent. If one is far out, the
cause is almost certainly a **silent market override** — see section 4.

### A warning you will see on `main`, already fixed on PR #157

`MDAGaussSeidel has reached its maximum number of unsuccessful iterations, but the
normalized residual norm 1.96e-05 is still above the tolerance 1e-10`

The Gauss-Seidel residual falls cleanly to 2e-5 and then pins there, identical to seven
digits, for thirteen further iterations. It is **not** a failed solve: every discipline
output is bit-identical between the last two sweeps, so the couplings have reached a
genuine fixed point. Nor is it caused by the constraint models of `constraints_rte.py`
— the residual is bit-identical with and without them.

The cause is `RPKElasticity` clipping the airfare inside `compute()`. The iterate says
*x*, the model computes with *clip(x)*, so the residual compares two different things
and can never reach zero. `demand.model: cagr_elasticity` is what puts that model in
the chain, which is why this scenario shows it.

**PR #157** (`fix/mda-residual-floor-and-global-disciplines`, commit `80fcbeda`) takes
the NaN sentinel out of the coupling vector and the clip out of the model, declaring
the bound through `MDAChain.set_bounds` instead. Measured on this exact case:

| | `main` | PR #157 |
|---|---|---|
| iterations | 21 (capped) | 11 |
| final residual | 1.961185e-05 | 3.257035e-11 |
| airfare 2050 | 0.10086028 | 0.10086028 |

Same answers, half the iterations, real convergence. So on `main` the results here are
trustworthy but the MDA does about twice the work it needs to, and every
finite-difference gradient pays that twice over. If you are going to run the
optimisation in earnest, run it on top of #157.

## 2. The optimisation problem

Identical to the published formulation: minimise the discounted total surplus loss over
10 design variables (5 reference years × 2 pathways) subject to G1–G6.

* **G1** carbon budget — `CarbonBudgetConstraint`, from `models_optim_complex`
* **G2** blend completeness, χ_B + χ_E ≤ 100
* **G3/G4** biomass and electricity availability
* **G5/G6** ramp-up, the least constraining of a rate and a volume limit (Eq. 12)

G2–G6 live in `constraints_rte.py`. Note that GEMSEO's `AutoPyDiscipline` requires each
`compute` to `return <name>`, never an expression, and the returned names must match the
constraint names registered on the scenario.

In [ ]:
def share_mda_across_functions(process, size=16):
    """One MDA solve per design point instead of one per function.

    MDF hands the objective and each constraint its own `MDOFunction`, and
    `set_differentiation_method("finite_differences")` differentiates each of them
    separately: seven sweeps over the same eleven design points, seven identical MDA
    solves each time. Measured on this problem, three SLSQP iterations cost 252 MDA
    solves where 33 are needed -- 607 s instead of 54 s.

    GEMSEO's own discipline cache does not absorb the repeats. Its key is a hash of all
    241 chain inputs, and three identical calls produce three entries even when the
    stored inputs compare byte-identical. Keying on the design variables -- the only
    thing the converged fixed point depends on -- does absorb them, with iterates
    identical to the uncached run and the objective agreeing to nine significant digits.

    An LRU of 16 covers a full function sweep (eleven points); without a bound the memo
    would retain every design point of the run, each holding 241 arrays.
    """
    from collections import OrderedDict

    mda = process.scenario.formulation.mda
    solve, memo = mda.execute, OrderedDict()

    def cached(input_data=None, **kwargs):
        if input_data is None:
            return solve(**kwargs)
        key = tuple((name, tuple(np.ravel(value))) for name, value in sorted(input_data.items()))
        if key in memo:
            memo.move_to_end(key)
        else:
            outputs = solve(input_data, **kwargs)
            memo[key] = {k: (v.copy() if hasattr(v, "copy") else v) for k, v in outputs.items()}
            while len(memo) > size:
                memo.popitem(last=False)
        return memo[key]

    mda.execute = cached
    return process


def setup_optimisation(process, max_iter=40, x0_biofuel=None, warm_start=False):
    """Configure the SLSQP problem of section 3.5 of the paper.

    Scenario parameters are already set by `build_process`; this only adds the
    design space, the objective and the constraints.

    `warm_start` trades reproducibility for speed and is off by default -- see the
    comment at the bottom of this function.
    """
    process.gemseo_settings["scenario_type"] = "MDO"
    process.gemseo_settings["formulation"] = "MDF"

    design_space = DesignSpace()
    design_space.add_variable(
        "generic_electrofuel_mandate_share_values_optim",
        # Lower bound held off zero: a pathway share returning to zero after being
        # positive gives 0/0 in its own share variable and the MDA dies with
        # "converged on NaN". This is the same reason biofuel carries a lower bound
        # of 2 -- numerics, not policy. 0.5 sits below the 1.2 % ReFuelEU 2030
        # sub-mandate, so it does not bind on any scenario of interest.
        size=5,
        lower_bound=[1e-5] * 5,
        upper_bound=[100] * 5,
        value=[0.5, 5.78, 15.03, 39.12, 55.41],
    )
    design_space.add_variable(
        "generic_biofuel_mandate_share_values_optim",
        size=5,
        lower_bound=[2] * 5,
        upper_bound=[100] * 5,
        value=[8.35, 21.24, 41.05, 43.81, 43.95],
    )
    process.gemseo_settings["design_space"] = design_space
    process.gemseo_settings["objective_name"] = "cumulative_total_surplus_loss_discounted_obj"

    process.create_gemseo_scenario()

    # Bring the objective into a range SLSQP is comfortable with.
    problem = process.scenario.formulation.optimization_problem
    problem.objective = problem.objective * 1e-10

    for constraint in [
        "aviation_carbon_budget_constraint",
        "blend_completeness_constraint",
        "electricity_trajectory_constraint",
        "biomass_trajectory_constraint",
        "electrofuel_use_growth_constraint",
        "biofuel_use_growth_constraint",
    ]:
        process.scenario.add_constraint(constraint, constraint_type="ineq")

    process.scenario.set_differentiation_method("finite_differences")
    process.gemseo_settings["algorithm"] = SLSQP_Settings(
        max_iter=max_iter,
        enable_progress_bar=True,
        ftol_abs=0.001,
        normalize_design_space=False,
    )

    # Design variables arrive as ndarray but are consumed as lists downstream.
    CustomDataConverter._list_names.update(process.scenario.get_optim_variable_names())

    # One MDA solve per design point rather than one per function. Roughly 11x here;
    # see the docstring above for why GEMSEO's own cache does not do this.
    share_mda_across_functions(process)

    if warm_start:
        # Start each MDA from the previous solve's couplings. The finite-difference
        # points sit ~1e-6 apart, so Gauss-Seidel needs about half the sweeps: measured
        # 10.3 -> 5.1 per solve, and the objective gradient 17.8 s -> 12.6 s.
        #
        # It is off by default because it makes f(x) depend on evaluation *order*. The
        # MDA stops as soon as the residual crosses tolerance, so the converged point
        # sits somewhere in a ball around the true fixed point and its position depends
        # on where the sweep started. Measured against a cold run, the gradient moves by
        # 9.4e-06 relative at tolerance 1e-10 and the SLSQP iterates visibly diverge.
        #
        # Tightening to 1e-12 shrinks that ball and brings the gradient back to 6.5e-07
        # relative (cosine 0.999999999999994) while keeping most of the gain. 1e-13 is
        # no better, so ~7e-07 is this problem's own floor rather than a tolerance one
        # can buy past.
        #
        # Use it for exploratory work -- scanning start points, checking whether a
        # budget is feasible at all. Published runs should stay reproducible.
        mda = process.scenario.formulation.mda
        mda.settings.warm_start = True
        for inner_mda in mda.inner_mdas:
            inner_mda.settings.warm_start = True
            inner_mda.settings.tolerance = 1e-12

    return process

### Running it

The starting point is the design-space warm start of the published notebook
(electrofuel at zero), **not** a converged optimum, so SLSQP legitimately starts from
an infeasible point and you will see `Optimization found no feasible point` for very
small `max_iter`. That is expected; it only means the run was cut off before feasibility
was reached.

One MDA takes a few seconds and each SLSQP iteration costs eleven of them
(finite differences over 10 variables), so budget accordingly. That eleven is only
true because `setup_optimisation` installs `share_mda_across_functions`; without it
MDF differentiates all seven functions separately and the true cost is 77 per
iteration, which is where the hour-long runs came from. **Start with a small
`max_iter` to confirm the problem is wired correctly before committing to a full run.**

`enable_progress_bar=True` is set above so you can see where it is — worth keeping.

In [ ]:
process_opt = build_process(optimisation=True, carbon_budget_share=3.8)
setup_optimisation(process_opt, max_iter=50, warm_start=True)

start = time.perf_counter()
process_opt.compute()
elapsed = time.perf_counter() - start

# Save the history first: extracting results is where a late failure would otherwise
# throw away the whole run.
process_opt.scenario.save_optimization_history("results_migrated_B26.hdf")

result = process_opt.scenario.get_result().optimization_result
problem = process_opt.scenario.formulation.optimization_problem

print(f"wall time     {elapsed:.1f} s")
print(f"evaluations   {len(problem.database)}")
print(f"obj / grad    {result.n_obj_call or 0} / {result.n_grad_call or 0}")
print(f"objective     {result.f_opt:.6f}")
print(f"feasible      {result.is_feasible}")
print(f"message       {result.message}")

In [ ]:
# x_opt is ordered as the design space was declared: electrofuel first, then biofuel.
x_opt = np.asarray(result.x_opt)
electrofuel_opt, biofuel_opt = x_opt[:5], x_opt[5:]

print(
    pd.DataFrame(
        {"biofuel (%)": biofuel_opt, "electrofuel (%)": electrofuel_opt},
        index=OPTIM_YEARS,
    )
    .round(2)
    .to_string()
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

refueleu_years = [2025, 2030, 2035, 2040, 2045, 2050]
ax.plot(
    refueleu_years,
    [2, 4.8, 15, 24, 27, 35],
    "--",
    color="tab:green",
    label="Biofuel — ReFuelEU (linear)",
)
ax.plot(
    refueleu_years,
    [0, 1.2, 5, 10, 15, 35],
    "--",
    color="tab:blue",
    label="Electrofuel — ReFuelEU (linear)",
)
ax.plot(OPTIM_YEARS, biofuel_opt, "-o", color="tab:green", label="Biofuel — optimised")
ax.plot(OPTIM_YEARS, electrofuel_opt, "-o", color="tab:blue", label="Electrofuel — optimised")

ax.set_xlabel("Year")
ax.set_ylabel("Drop-in fuel share (%)")
ax.set_title("Optimised blending mandate vs ReFuelEU (compare with Figure 7)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

## 3. Diagnosing a run

Every evaluation is kept in the optimisation database, and `save_optimization_history`
wrote it to HDF above — so a run can be dissected afterwards without re-optimising it.
At tens of minutes a run, save first and diagnose from the file.

Note the granularity: rows are *evaluations*, not iterations. With 10 design variables and
finite differences that is roughly eleven rows per gradient, so the history is far longer
than the iteration count.

In [ ]:
from gemseo.algos.optimization_problem import OptimizationProblem

problem = OptimizationProblem.from_hdf("results_migrated_B26.hdf")

print(f"{len(problem.database)} evaluations")
print("recorded functions:", problem.database.get_function_names())

### The whole history as one table

`get_history_array` returns the array, its column names and the design variables
together — the fastest way to see objective, constraints and x side by side.

In [ ]:
# Database.get_history_array() cannot stack this problem's history: the objective is
# stored 0-dimensional while the constraints are 5-vectors, and line-search entries hold
# only the objective -- its internal hstack then sees mixed ranks and raises
# "all the input arrays must have same number of dimensions". Build the frame directly.
# This also gives one named column per constraint component rather than opaque indices.
x_names = problem.design_space.get_indexed_variable_names()

rows = []
for x in problem.database.get_x_vect_history():
    entry = problem.database[x]
    row = {}
    for name, value in entry.items():
        # ravel, not atleast_1d: some values are stored 2-D as well as 0-D.
        values = np.asarray(value, dtype=float).ravel()
        if values.size == 1:
            row[name] = float(values[0])
        else:
            row.update({f"{name}[{i}]": float(v) for i, v in enumerate(values)})
    row.update(dict(zip(x_names, np.asarray(x, dtype=float))))
    rows.append(row)

# Evaluations where the optimiser only needed the objective leave the constraints NaN.
history = pd.DataFrame(rows)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
print(f"{len(history)} evaluations x {history.shape[1]} columns")
print(history.tail(15))

history.to_csv("history_B26.csv", index=False)

### Did it ever reach the feasible set?

Usually the fastest read on a failed run. If the worst violation never crosses zero the
optimiser never found the feasible set at all; if it crosses and then leaves again,
feasibility was traded for objective and the penalty balance is the suspect.

In [ ]:
constraint_names = [c.name for c in problem.constraints]

worst = []
for x in problem.database.get_x_vect_history():
    entry = problem.database[x]
    present = [float(np.max(entry[n])) for n in constraint_names if n in entry]
    worst.append(max(present) if present else np.nan)
worst = np.array(worst)

reached = np.where(worst <= 1e-4)[0]
print(f"worst violation: first {worst[0]:+.4f}  ->  last {worst[-1]:+.4f}")
print(f"minimum reached: {np.nanmin(worst):+.4f}")
print("first feasible evaluation:", int(reached[0]) if reached.size else "never")

In [ ]:
# One constraint across the run, with the design vectors that produced each value.
values, xs = problem.database.get_function_history(
    "biofuel_use_growth_constraint", with_x_vect=True
)
print("biofuel ramp constraint, last 5 evaluations:")
print(np.asarray(values)[-5:])

### Plots

`ConstraintsHistory` is the one for this question: a panel per constraint with the
feasible region shaded, so a constraint that never comes down is visible at a glance.
`ObjConstrHist` overlays objective against violation, which is what shows trading.
`execute_post` accepts the HDF path directly, so the reload above is optional.

In [ ]:
from gemseo import execute_post
from gemseo.settings.post import ConstraintsHistory_Settings

execute_post(
    problem,
    ConstraintsHistory_Settings(
        constraint_names=constraint_names, save=True, file_path="constraints_B26"
    ),
)

# ObjConstrHist is deliberately not used: it formats the worst violation on a log scale
# and raises "cannot convert float NaN to integer" as soon as one evaluation lacks a
# constraint value -- which is every line-search point. Plot the same thing directly.
objective_column = next(c for c in history.columns if "surplus" in c)

fig, ax_obj = plt.subplots(figsize=(8, 4))
ax_obj.plot(history.index, history[objective_column], color="tab:blue")
ax_obj.set_xlabel("evaluation")
ax_obj.set_ylabel("objective", color="tab:blue")
ax_obj.tick_params(axis="y", labelcolor="tab:blue")

ax_viol = ax_obj.twinx()
ax_viol.plot(history.index, worst, color="tab:red")
ax_viol.axhline(0.0, color="tab:red", linestyle=":", linewidth=1)
ax_viol.set_ylabel("worst constraint violation", color="tab:red")
ax_viol.tick_params(axis="y", labelcolor="tab:red")

ax_obj.set_title("Objective against feasibility over the run")
plt.tight_layout()

### When the run ends infeasible

GEMSEO always populates `constraint_values` on the result, but only *prints* them when
there are fewer than 20 — with 26 constraints the listing is silently dropped and you are
left with the verdict alone. Read them off the object instead.

`ineq_tolerance` defaults to **1e-4**, so `is_feasible` can be False on a violation too
small to matter. Sort by magnitude before concluding anything.

In [ ]:
violated = sorted(
    ((float(np.max(v)), k) for k, v in (result.constraint_values or {}).items()),
    reverse=True,
)
print(f"feasible: {result.is_feasible}   (ineq_tolerance = 1e-4)")
for value, name in violated:
    mark = "VIOLATED" if value > 1e-4 else "ok"
    print(f"  {name:45s} {value:+.6f}  {mark}")

## 4. Optional — the collapsed single-market variant

Same scenario on one aggregated passenger market: 90 disciplines instead of 100, with
results identical to 7 significant figures. The split is inert here because the paper
gives all three passenger markets the same growth and the same efficiency assumptions.

In [ ]:
process_1m = build_process(config="config_1m.yaml")
set_mandate(
    process_1m, biofuel=[4.8, 15.0, 24.0, 27.0, 35.0], electrofuel=[1.2, 5.0, 10.0, 15.0, 35.0]
)
process_1m.compute()

print(f"{len(process_1m.mda_chain.disciplines)} disciplines")
print(f"airfare 2050  4 markets {process.data['vector_outputs']['airfare_per_rpk'].loc[2050]:.8f}")
print(
    f"              1 market  {process_1m.data['vector_outputs']['airfare_per_rpk'].loc[2050]:.8f}"
)

## 5. Migration notes — the traps

**Market parameters defined in `markets.yaml` silently win over the input JSON.** Four
of the paper's assumptions were being overridden with no error raised. Each is now set
in `markets_rte.yaml`:

| parameter | paper | default that was winning | effect if missed |
|---|---|---|---|
| CAGR | 2.2 % | 3.0 % | RPK 2050 +23.6 % |
| partitioning shares | EU (AeroSCOPE) | global | wrong traffic mix |
| efficiency gain | 1.35 %/yr | 2.0 %/yr | cumulative CO₂ −7 % |
| load factor 2050 | 89 % | 85 % | **surplus loss flips sign** |

The last one is the dangerous one: with it wrong the objective read +59.5 Bn€ instead
of −10.4 Bn€, and nothing warned. Anything market-scoped belongs in the markets file,
not in `inputs.json`.

**Other things worth knowing**

* `_share_2019` keys are now `_share_last_historical_year`. This one *does* raise.
* A pathway series written `years: []` will not span the historic range, and
  `scenario_cost.py` reads the kerosene emission factor at `prospection_start_year - 1`.
  Give every pathway an explicit span, e.g. `years: [2000, 2050]`.
* `setup_mda()` uses `tolerance=1e-10, max_mda_iter=200` on main against `1e-7` on
  `optim_backwards`, so a bare MDA timing comparison between the two branches is not
  like-for-like. The MDO path uses `1e-4` on both and *is* comparable. For this scenario
  1e-7 and 1e-10 give the same answer to six decimals.
* `kerosene_selectivity` is set to 1.0 on both pathways, mapping the old model's
  efficiencies as jet-specific. Worth re-checking against the #158 selectivity fix.

**Cross-check before trusting an optimum.** The MDA outputs were validated against
`optim_backwards`, but the *constraint* values were not. At the warm-start point the
migrated `aviation_carbon_budget_constraint` evaluates to +0.51, and the arithmetic
relating it to `aviation_carbon_budget` (20.78) and `gross_carbon_budget` (1135) does
not obviously reduce to `(cumulative - budget) / budget`. Before reading anything into
an optimised mandate, evaluate the same design point on both branches and confirm the
six constraints agree in scale — a constraint scaled differently moves the optimum
without moving any MDA output.

The other five constraints are computed by the disciplines but do not surface in
`vector_outputs` / `float_outputs` (they are list-valued). GEMSEO reads them directly
from the disciplines, which is why the scenario builds; to inspect them outside the
optimiser, read them off the discipline rather than the process data.

**Not yet established.** Whether the migrated problem converges in fewer or more
evaluations than `optim_backwards`, and whether it reaches the same optimum. Runs on
both branches were still going at 47 minutes when they were stopped, on a machine that
was not idle. Comparing `n_obj_call` / `n_grad_call` is the load-independent way to
settle it.